In [ ]:
# Fetch the data from different sources

# ICOS-Carbon-data- portal: https://github.com/ICOS-Carbon-Portal/data?tab=readme-ov-file#downloading-originals-programmatically

# Create an ICOS Carbon portal account: https://cpauth.icos-cp.eu/login/


In [ ]:
import json
import os
import zipfile

import polars as pl
from icoscp_core.icos import bootstrap

In [ ]:
token_file_path = "../tokens/cpauthToken_auth_conf.json"

In [ ]:
def get_icoscp_credentials(token_file_path):
    """Load the authentication token from the provided JSON file."""
    with open(token_file_path, "r") as f:
        credentials = json.load(f)
    return credentials


credentials = get_icoscp_credentials(token_file_path)

meta, data = bootstrap.fromCredentials(credentials["username"], credentials["password"])

In [ ]:
# from icoscp_core.icos import meta

# fetches the list of known data types, including metadata associated with them
all_datatypes = meta.list_datatypes()

# data types with structured data access
previewable_datatypes = [dt for dt in all_datatypes if dt.has_data_access]

datatypes = [{"Description": dt.label, "uri": dt.uri} for dt in previewable_datatypes]

df = pl.DataFrame(datatypes)

df.write_csv("../data/intermediate/previewable_datatypes.csv")

df.head()

In [ ]:
# from icoscp_core.icos import meta

# Ecosystem final quality (L2) product in ETC-Archive format - release 2025-1
collection_uri = "https://meta.icos-cp.eu/collections/1-HA2r4l5QUjAgQr5CCEfJe3"

""" 
The Level 2 Ecosystem collection consists of many zipped parts for 
individual stations and components. Examples (as shown on the collection page) 
include:
*_FLUXES_L2.zip — Flux measurements (GPP, NEE, latent/sensible heat)
*_METEO_L2.zip — Meteorological variables (radiation, temperature, humidity, 
    wind, etc.)
*_AUXDATA_L2.zip — Ancillary/biometric measurements

"""

collection_meta = meta.get_collection_meta(collection_uri)

members = collection_meta.members

In [ ]:
collection_uri = "http://meta.icos-cp.eu/resources/stations/AS_HTM"
collection_meta = meta.get_station_meta(collection_uri)
print(dir(collection_meta))

In [ ]:
# Extract the uri for meteorological data

meteorological_urls = []
for mem in members:
    if hasattr(mem, "name") and hasattr(mem, "res"):
        name = mem.name
        if isinstance(name, str) and "METEO" in name:
            meteorological_urls.append(mem.res)
            # print(mem.res, name)

# Extract uri's for fluxes
flux_urls = []
for mem in members:
    if hasattr(mem, "name") and hasattr(mem, "res"):
        name = mem.name
        if isinstance(name, str) and "FLUXES_L2" in name:
            flux_urls.append(mem.res)
            # print(mem.res, name)

# Extract uri's for ancillary measurememnts
aux_urls = []
for mem in members:
    if hasattr(mem, "name") and hasattr(mem, "res"):
        name = mem.name
        if isinstance(name, str) and "AUXDATA" in name:
            aux_urls.append(mem.res)
            print(mem.res, name)

In [ ]:
n20_urls = []
for mem in members:
    if hasattr(mem, "name") and hasattr(mem, "res"):
        name = mem.name
        if isinstance(name, str) and "N2O" in name:
            n20_urls.append(mem.res)
            print(mem.res, name)

In [ ]:
from icoscp_core.icos import ATMO_STATION, meta

# fetch lists of stations, with basic metadata
icos_stations = meta.list_stations()
atmo_stations = meta.list_stations(ATMO_STATION)
all_known_stations = meta.list_stations(False)

# get detailed metadata for a station
htm_uri = "http://meta.icos-cp.eu/resources/stations/AS_HTM"
htm_station_meta = meta.get_station_meta(htm_uri)

htm_station_meta

In [ ]:
# Creating a table for ICOS stations
icos_stations = meta.list_stations()
station_data = []
schema = ["Id", "Label", "Name", "Country code", "Lat", "Lon"]

for station in icos_stations:
    station_data.append(
        {
            "Id": station.type_uri,
            "Label": station.label,
            "Name": station.name,
            "Country code": station.country_code,
            "Lat": station.lat,
            "Lon": station.lon,
        }
    )

station_data = pl.DataFrame(
    station_data,
    schema=schema,
)

station_data

In [ ]:
icos_stations

In [ ]:
from icoscp_core.metaclient import SamplingHeightFilter, SizeFilter, TimeFilter

# list data objects with basic metadata
# a contrived, complicated example to demonstrate the possibilities
# all the arguments are optional
# see the Python help for the method for more details
filtered_atc_co2 = meta.list_data_objects(
    datatype=[
        "http://meta.icos-cp.eu/resources/cpmeta/atcCo2L2DataObject",
        "http://meta.icos-cp.eu/resources/cpmeta/atcCo2NrtGrowingDataObject",
    ],
    station="http://meta.icos-cp.eu/resources/stations/AS_GAT",
    filters=[
        TimeFilter("submTime", ">", "2016-07-01T12:00:00Z"),
        TimeFilter("submTime", "<", "2018-07-10T12:00:00Z"),
        SizeFilter(">", 5000),
        SamplingHeightFilter("=", 216),
    ],
    include_deprecated=True,
    order_by="fileName",
    # limit = 50
)

filtered_atc_co2

In [ ]:
from icoscp_core.icos import meta

station_uri = "http://meta.icos-cp.eu/resources/stations/AS_HTM"

# List ALL data objects associated with this station
dobjs = meta.list_data_objects(station=station_uri, limit=10000)

# Extract URIs
dobj_uris = [dobj.uri for dobj in dobjs]

fluxes_l2 = [d for d in dobjs if "FLUXES" in d.filename]

len(dobjs)

In [ ]:
dobjs = meta.list_data_objects(station=station_uri, limit=10000)
for d in dobjs:
    print(d.uri, d.filename)

In [ ]:
from icoscp_core.icos import station_class_lookup

htm_uri = "http://meta.icos-cp.eu/resources/stations/AS_HTM"
htm_class = station_class_lookup()[htm_uri]

htm_class

In [ ]:
htm_uri = "http://meta.icos-cp.eu/resources/stations/AS_HTM"
htm_station_meta = meta.get_station_meta(htm_uri)

htm_station_meta.__dict__

In [ ]:
from icoscp_core.icos import meta

station_uri = "http://meta.icos-cp.eu/resources/stations/AS_HTM"

all_objects = []
offset = 0
batch_size = 1000  # fetch 1000 at a time

while True:
    batch = meta.list_data_objects(station=station_uri, limit=batch_size, offset=offset)
    if not batch:
        break
    all_objects.extend(batch)
    offset += batch_size
    # print(f"Fetched {len(all_objects)} objects so far...")

print(f"Total objects fetched: {len(all_objects)}")

In [ ]:
all_objects[0].__dict__

In [ ]:
all_objects

In [ ]:
from icoscp_core.icos import meta

# List all data objects (returns DobjSpecLite objects)
all_objects = meta.list_data_objects()

# Extract URIs in the exact format you want
all_uris = [dobj.uri for dobj in all_objects]

# Inspect
len(all_uris), all_uris[:5]

In [ ]:
all_objects

In [ ]:
all_objects[0].datatype_uri

In [ ]:
dobj_uris = [
    "https://meta.icos-cp.eu/objects/E2MVHezJQReXShfzPtBVlVwS",
    "https://meta.icos-cp.eu/objects/udtdN1NfB3YNvclaEm5RpbCd",
]

# dobj_uris = ["https://meta.icos-cp.eu/objects/wFrOPLqm3CcOZpUmxfbPTGqo"]

folder_path = "../data/raw/ICOS/"

unsucessful_count = 0

for idx, dobj_uri in enumerate(meteorological_urls[0:1]):
    try:
        filename = data.save_to_folder(dobj_uri, folder_path)

        zip_file_path = os.path.join(folder_path, filename)
        print(filename, zip_file_path, zip_file_path[:-4])

        # Extract the ZIP file
        with zipfile.ZipFile(zip_file_path, "r") as zip_ref:
            zip_ref.extractall(folder_path)

    except Exception as e:
        print(f"Error downloading file: {e} ----- {idx}----- {dobj_uri}")

In [ ]:
meteorological_urls[0]

In [ ]:
dobj = meta.get_dobj_meta("https://meta.icos-cp.eu/objects/hujSGCfmNIRdxtOcEvEJLxGM")
coll_info = dobj.parentCollections[0]
coll_label = coll_info.label
coll_2010 = meta.get_collection_meta(coll_info.uri)
coll_evapo_info = coll_2010.parentCollections[0]
coll_evapo = meta.get_collection_meta(coll_evapo_info.uri)
evapo_years = coll_evapo.members
top_coll_info = coll_evapo.parentCollections[0]
top_coll = meta.get_collection_meta(top_coll_info.uri)
top_subcols = top_coll.members

In [ ]:
dobj.__dict__

In [ ]:
dobj.parentCollections

In [ ]:
dobj.parentCollections[0]

In [ ]:
meta.get_collection_meta(dobj.parentCollections[0].uri)

In [ ]:
coll_2010 = meta.get_collection_meta(dobj.parentCollections[0].uri)

coll_2010.__dict__

In [ ]:
coll_evapo = meta.get_collection_meta(coll_2010.parentCollections[0].uri)

coll_evapo.__dict__

In [ ]:
evapo_01_21 = meta.get_collection_meta(coll_evapo.parentCollections[0].uri)

evapo_01_21.__dict__